In [ ]:
import os, random, json
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score


In [2]:

GRAPH_DIR = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
OUT_DIR = os.path.join(GRAPH_DIR, "pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

HDIM = 128
OUTDIM = 128
EPOCHS = 6
BATCH_SIZE = 2048
NEG_RATIO = 1
PATH_REG_WEIGHT = 0.5
ENSEMBLE_SIZE = 2
MC_RUNS = 30
TOP_K = 50


In [3]:

def safe_read_lines(path):
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return [ln.rstrip("\n") for ln in f]

def read_csv_triples(path):
    if not os.path.exists(path):
        return []
    df = pd.read_csv(path, header=None, dtype=str)
    if df.shape[1] < 3:
        return []
    return [(r[0].strip(), r[1].strip(), r[2].strip()) for r in df.values if len(r)>=3]


In [4]:

edge_index = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt")).long()
edge_type = torch.load(os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_type.pt")).long() if os.path.exists(os.path.join(GRAPH_DIR,"edge_type.pt")) else None
num_nodes = int(edge_index.max().item()) + 1

entities_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt"))
relation_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt"))

train_triples = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv"))
val_triples   = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv"))
test_triples  = read_csv_triples(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv"))


In [17]:
entities_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt")
entities_lines = safe_read_lines(entities_path)

ent2type = {}
for line in entities_lines:
    ent_id = line.strip()
    if "Compound::" in ent_id:
        ent2type[ent_id] = "compound"
    elif "Gene::" in ent_id:
        ent2type[ent_id] = "gene"
    elif "Disease::" in ent_id:
        ent2type[ent_id] = "disease"
    else:
        ent2type[ent_id] = "other"

all_entities = list(ent2type.keys())
ent2idx = {e: i for i, e in enumerate(all_entities)}
idx2ent = {i: e for e, i in ent2idx.items()}

print(f"Loaded {len(ent2type)} entities: "
      f"{sum(1 for v in ent2type.values() if v=='compound')} compounds, "
      f"{sum(1 for v in ent2type.values() if v=='gene')} genes, "
      f"{sum(1 for v in ent2type.values() if v=='disease')} diseases.")


Loaded 94046 entities: 23125 compounds, 38826 genes, 4023 diseases.


In [ ]:
all_entities = list(ent2type.keys())  # all IDs as strings
ent2idx = {e:i for i,e in enumerate(all_entities)}
idx2ent = {i:e for e,i in ent2idx.items()}

num_nodes = len(all_entities)  # number of embeddings


In [19]:

def build_cd_pairs(triples):
    pos = []
    for h,r,t in triples:
        h_type = ent2type.get(h,"other")
        t_type = ent2type.get(t,"other")
        if (h_type=="compound" and t_type=="disease") or "treat" in r.lower():
            pos.append((ent2idx[h], ent2idx[t]))  # map to embedding index
    return list(set(pos))

# Mechanistic pairs:
comp_gene_pairs = set()
gene_disease_pairs = set()
for h,r,t in train_triples:
    h_type = ent2type.get(h,"other"); t_type = ent2type.get(t,"other")
    if h_type=="compound" and t_type=="gene": comp_gene_pairs.add((ent2idx[h], ent2idx[t]))
    if h_type=="gene" and t_type=="disease": gene_disease_pairs.add((ent2idx[h], ent2idx[t]))


In [8]:
def mechanistic_path_loss(embs, comp_gene, gene_disease, max_samples=1024):
    if len(comp_gene)==0 or len(gene_disease)==0:
        return torch.tensor(0., device=embs.device, requires_grad=True)
    cg = random.sample(list(comp_gene), min(len(comp_gene), max_samples))
    gd = random.sample(list(gene_disease), min(len(gene_disease), max_samples))
    comp_to_genes = defaultdict(list)
    gene_to_diseases = defaultdict(list)
    for c,g in cg: comp_to_genes[c].append(g)
    for g,d in gd: gene_to_diseases[g].append(d)
    triplets=[]
    for c,genes in comp_to_genes.items():
        for g in genes:
            for d in gene_to_diseases.get(g, []):
                triplets.append((c,g,d))
    if len(triplets)==0:
        return torch.tensor(0., device=embs.device, requires_grad=True)
    c_idx = torch.tensor([t[0] for t in triplets], dtype=torch.long, device=embs.device)
    g_idx = torch.tensor([t[1] for t in triplets], dtype=torch.long, device=embs.device)
    d_idx = torch.tensor([t[2] for t in triplets], dtype=torch.long, device=embs.device)
    c_v, g_v, d_v = embs[c_idx], embs[g_idx], embs[d_idx]
    pred_cd = torch.sum(F.normalize(c_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    cg_sim = torch.sum(F.normalize(c_v,dim=1)*F.normalize(g_v,dim=1), dim=1)
    gd_sim = torch.sum(F.normalize(g_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    mech = torch.min(cg_sim, gd_sim)
    margin = 0.1
    loss = F.relu(margin + mech - pred_cd).mean()
    return loss


In [9]:

def train_model(model, pairs, edge_index, comp_gene_pairs, gene_disease_pairs,
                lr=1e-3, epochs=EPOCHS, batch_size=BATCH_SIZE):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(pairs)
    for epoch in range(epochs):
        random.shuffle(pairs)
        total_loss = 0
        model.train()
        for i in range(0, n, batch_size):
            batch = pairs[i:i+batch_size]
            comps = [c for c,d,l in batch]
            dises = [d for c,d,l in batch]
            labels = torch.tensor([l for _,_,l in batch], dtype=torch.float, device=DEVICE)
            embs = model(edge_index.to(DEVICE))
            c_embs = embs[torch.tensor(comps, device=DEVICE)]
            d_embs = embs[torch.tensor(dises, device=DEVICE)]
            logits = (c_embs*d_embs).sum(dim=1)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            # --- mechanistic path reg ---
            path_loss = mechanistic_path_loss(embs, comp_gene_pairs, gene_disease_pairs)
            total_batch_loss = loss + PATH_REG_WEIGHT*path_loss
            opt.zero_grad()
            total_batch_loss.backward()
            opt.step()
            total_loss += total_batch_loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f}")



In [10]:
@torch.no_grad()
def predict_with_uncertainty(model, edge_index, pairs, mc_runs=MC_RUNS):
    model.eval()
    comps = [c for c,d,l in pairs]
    dises = [d for c,d,l in pairs]
    logits_list = []
    for _ in range(mc_runs):
        embs = model(edge_index.to(DEVICE))
        c_embs = embs[torch.tensor(comps, device=DEVICE)]
        d_embs = embs[torch.tensor(dises, device=DEVICE)]
        logits = (c_embs*d_embs).sum(dim=1)
        logits_list.append(torch.sigmoid(logits).cpu().numpy())
    all_logits = np.stack(logits_list, axis=0)
    mean_preds = all_logits.mean(axis=0)
    std_preds = all_logits.std(axis=0)
    return mean_preds, std_preds


In [11]:
def save_ranked_csv(pairs, mean_preds, std_preds, outpath):
    df = pd.DataFrame({
        "compound": [c for c,d,l in pairs],
        "disease": [d for c,d,l in pairs],
        "score": mean_preds,
        "uncertainty": std_preds
    })
    df.sort_values("score", ascending=False, inplace=True)
    df.to_csv(outpath, index=False)
    print(f"Saved ranked CSV: {outpath}")



In [20]:
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, RGCNConv, TransformerConv

In [26]:
entities_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt")
entities_lines = safe_read_lines(entities_path)

ent2type = {}
for line in entities_lines:
    parts = line.strip().split("\t")
    if len(parts) >= 2:
        ent_id, ent_type = parts[0], parts[1].lower()  
        if "compound" in ent_type:
            ent2type[ent_id] = "compound"
        elif "gene" in ent_type:
            ent2type[ent_id] = "gene"
        elif "disease" in ent_type:
            ent2type[ent_id] = "disease"
        else:
            ent2type[ent_id] = "other"

# map all entities to integer indices
all_entities = list(ent2type.keys())
ent2idx = {e:i for i,e in enumerate(all_entities)}
idx2ent = {i:e for e,i in ent2idx.items()}


In [33]:
for h,r,t in train_triples + val_triples + test_triples:
    for ent in [h,t]:
        if ent not in ent2type:
            if "Compound::" in ent:
                ent2type[ent] = "compound"
            elif "Gene::" in ent:
                ent2type[ent] = "gene"
            elif "Disease::" in ent:
                ent2type[ent] = "disease"
            else:
                ent2type[ent] = "other"

# remap indices
all_entities = list(ent2type.keys())
ent2idx = {e:i for i,e in enumerate(all_entities)}
idx2ent = {i:e for e,i in ent2idx.items()}

In [32]:
def build_cd_pairs(triples):
    pairs = []
    for h, r, t in triples:
        h_type = ent2type.get(h, "")
        t_type = ent2type.get(t, "")
        if ("compound" in h_type and "disease" in t_type) or "treat" in r.lower():
            pairs.append((ent2idx[h], ent2idx[t], 1))  # positive sample
    return pairs

train_pairs = build_cd_pairs(train_triples)
val_pairs   = build_cd_pairs(val_triples)
test_pairs  = build_cd_pairs(test_triples)

In [35]:
class GCNModel(nn.Module):
    def __init__(self, num_nodes, hdim=128):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hdim)
        self.conv1 = GCNConv(hdim, hdim)
        self.conv2 = GCNConv(hdim, hdim)

    def forward(self, edge_index):
        x = self.emb.weight.to(edge_index.device)  # ensures same device
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x


class GATModel(nn.Module):
    def __init__(self, num_nodes, hdim=HDIM, heads=4):
        super().__init__()
        self.device = DEVICE
        self.emb = nn.Embedding(num_nodes, hdim)
        self.conv1 = GATConv(hdim, hdim//heads, heads=heads, dropout=0.2)
        self.conv2 = GATConv(hdim, hdim//heads, heads=heads, dropout=0.2)
    def forward(self, edge_index):
        x = self.emb.weight
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class GraphSAGEModel(nn.Module):
    def __init__(self, num_nodes, hdim=HDIM):
        super().__init__()
        self.device = DEVICE
        self.emb = nn.Embedding(num_nodes, hdim)
        self.conv1 = SAGEConv(hdim, hdim)
        self.conv2 = SAGEConv(hdim, hdim)
    def forward(self, edge_index):
        x = self.emb.weight
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class RGCNModel(nn.Module):
    def __init__(self, num_nodes, num_rels, hdim=HDIM):
        super().__init__()
        self.device = DEVICE
        self.emb = nn.Embedding(num_nodes, hdim)
        self.conv1 = RGCNConv(num_nodes, hdim, num_rels, num_bases=30)
        self.conv2 = RGCNConv(hdim, hdim, num_rels, num_bases=30)
    def forward(self, edge_index, edge_type=None):
        x = self.emb.weight
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = self.conv2(x, edge_index, edge_type)
        return x

class TransformerGNNModel(nn.Module):
    def __init__(self, num_nodes, hdim=HDIM, heads=4):
        super().__init__()
        self.device = DEVICE
        self.emb = nn.Embedding(num_nodes, hdim)
        self.conv1 = TransformerConv(hdim, hdim//heads, heads=heads)
        self.conv2 = TransformerConv(hdim, hdim//heads, heads=heads)
    def forward(self, edge_index):
        x = self.emb.weight
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

In [37]:
def compute_loss(embs, pairs, comp_gene_pairs, gene_disease_pairs, device=DEVICE):
    """
    Compute the BCE loss for positive/negative pairs + mechanistic path regularization.
    
    Args:
        embs: Node embeddings (num_nodes x hidden_dim)
        pairs: List of tuples (compound_idx, disease_idx, label)
        comp_gene_pairs: Set of (compound_idx, gene_idx)
        gene_disease_pairs: Set of (gene_idx, disease_idx)
    """
    comps = [c for c,d,l in pairs]
    dises = [d for c,d,l in pairs]
    labels = torch.tensor([l for _,_,l in pairs], dtype=torch.float, device=device)
    
    c_embs = embs[torch.tensor(comps, device=device)]
    d_embs = embs[torch.tensor(dises, device=device)]
    logits = (c_embs * d_embs).sum(dim=1)
    
    bce_loss = F.binary_cross_entropy_with_logits(logits, labels)
    path_loss = mechanistic_path_loss(embs, comp_gene_pairs, gene_disease_pairs)
    
    total_loss = bce_loss + PATH_REG_WEIGHT * path_loss
    return total_loss


In [22]:
def train_model(model, train_pairs, edge_index, comp_gene_pairs, gene_disease_pairs, epochs=50, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # Forward pass
        embeddings = model(edge_index)
        loss = compute_loss(embeddings, train_pairs, comp_gene_pairs, gene_disease_pairs, criterion)

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.4f}")


In [36]:
if __name__ == "__main__":
    models = {
        "GCN": GCNModel(num_nodes),
        "GAT": GATModel(num_nodes),
        "GraphSAGE": GraphSAGEModel(num_nodes),
        "RGCN": RGCNModel(num_nodes, num_rels=len(relation_lines)),
        "Transformer": TransformerGNNModel(num_nodes)
    }

    for name, model in models.items():
        print(f"\n=== Training {name} ===")
        train_model(model, train_pairs, edge_index, comp_gene_pairs, gene_disease_pairs, epochs=100)
        mean_preds, std_preds = predict_with_uncertainty(model, edge_index, test_pairs)
        out_path = os.path.join(OUT_DIR, f"{name}_ranked_test.csv")
        save_ranked_csv(test_pairs, mean_preds, std_preds, out_path)



=== Training GCN ===


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_CUDA_mm)